# Filter aggregated classification results

Keep the requested BERTopic and indicator configuration, select and rename the evaluation metrics, and add an average across all models for every account-relations configuration.

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
def find_project_root(start_path):
    """Find the project root containing AGENTS.md.

    Args:
        start_path: Directory from which to begin the search.

    Returns:
        Path to the Backbone-Graph project root.
    """
    start_path = Path(start_path).resolve()
    for directory in (start_path, *start_path.parents):
        if (directory / "AGENTS.md").exists():
            return directory
    raise FileNotFoundError("Could not find the Backbone-Graph project root.")

In [ ]:
project_root = find_project_root(Path.cwd())
csv_path = (
    project_root
    / "results"
    / "labeled_datasets"
    / "step_4_inter_intra_classification"
    / " aggregated"
    / "top_percentage_100_iran_1_uae_venezuela_china_1_cuba_russia_1"
    / "aggregated_classification_sweep_results.csv"
)

classification_df = pd.read_csv(csv_path)

In [ ]:
filtered_df = (
    classification_df.loc[
        classification_df["indicators_configuration"].eq("All")
        & classification_df["topic_col"].eq("BERTopic_topic_256"),
        [
            "model",
            "account_relations_configuration",
            "all_authors_auc",
            "all_authors_auc_pr",
        ],
    ]
    .rename(
        columns={
            "all_authors_auc": "roc_auc",
            "all_authors_auc_pr": "pr_auc",
        }
    )
    .reset_index(drop=True)
)

filtered_df

In [ ]:
metric_columns = ["roc_auc", "pr_auc"]
average_rows = (
    filtered_df.groupby(
        "account_relations_configuration",
        as_index=False,
    )[metric_columns]
    .mean()
)
average_rows.insert(0, "model", "all models average")

results_df = pd.concat(
    [filtered_df, average_rows],
    ignore_index=True,
)

results_df

In [ ]:
output_dir = (
    project_root.parent
    / "paper_overleaf"
    / "ComplexNetworksTemplate"
    / "figs"
)
output_dir.mkdir(parents=True, exist_ok=True)
latex_path = output_dir / "filtered_aggregated_classification_results.tex"

results_df.to_latex(
    latex_path,
    index=False,
    float_format="%.3f",
    column_format="llrr",
    escape=True,
)

print(f"Saved LaTeX table: {latex_path}")